In [4]:
import os
import tempfile
import shutil
import json
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser

c:\Users\acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\acer\AppData\Local\Temp\ipykernel_2536\2383170035.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.generic import GenericLoader


# 1. Test Cloning

In [12]:
test_repo_url = "https://github.com/lighthouse-labs/todo-list-js-exercise.git" 
temp_dir = tempfile.mkdtemp()
print(f"Cloning into {temp_dir}...")
Repo.clone_from(test_repo_url, temp_dir)


Cloning into C:\Users\acer\AppData\Local\Temp\tmpec627czq...


<git.repo.base.Repo 'C:\\Users\\acer\\AppData\\Local\\Temp\\tmpec627czq\\.git'>


# 2. Test Loading & Parsing

In [14]:
loader = GenericLoader.from_filesystem(
    temp_dir,
    glob="**/*",
    suffixes=[".py", ".js", ".ts", ".md"],
    exclude=["**/node_modules/**", "**/.git/**"],
    parser=LanguageParser()
)
docs = loader.load()
print(f"Loaded {len(docs)} documents.")

Loaded 4 documents.


# 3. Test Chunking

In [16]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=500, chunk_overlap=50
)
chunks = python_splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks!")

Created 5 chunks!



# 4. Save to JSON so Notebook 02 (Vector DB) can use it later

In [17]:
output_data = [{"page_content": c.page_content, "metadata": c.metadata} for c in chunks]
with open("data/chunks.json", "w") as f:
    json.dump(output_data, f)
print("Saved chunks to data/chunks.json")

Saved chunks to data/chunks.json



# 5. Cleanup the cloned repo

In [19]:
shutil.rmtree(temp_dir, ignore_errors=True)